# Route Resilience — GPU training (Colab / Kaggle)

Trains the **baseline U-Net** (Dice+Focal) and the **SegFormer-B2 + clDice** model,
then compares them so the topology (clDice/connectivity/APLS) advantage is visible.

**Before running:** Runtime → Change runtime type → **GPU** (T4 is fine).

We train on cloud GPUs because local dev has no GPU; the code is device-agnostic,
so these are the same scripts that run locally on CPU. Download the resulting
`*.pth` checkpoints into local `artifacts/checkpoints/` to run the dashboard.

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/Rayyan-mohammed/Urban-Route-Resilience.git
%cd Urban-Route-Resilience

In [ ]:
# Colab/Kaggle already ship a CUDA torch; install the rest + the package.
!pip install -q segmentation-models-pytorch timm albumentations omegaconf rich \
    rasterio geopandas osmnx scikit-image networkx pyproj scipy
!pip install -q -e .

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Build the dataset (OSMnx road masks, 4 terrains)
Reproduces the geo-referenced mask tiles + terrain-stratified split. Swap in real
SpaceNet/DeepGlobe/Cartosat imagery by setting `image_path` in the manifest later.

In [ ]:
!python scripts/build_dataset.py

## 2. Train the baseline (U-Net, Dice+Focal) — the control

In [ ]:
!python scripts/train.py -o train.num_workers=2 -o train.epochs=40

## 3. Train SegFormer-B2 + clDice — the topology-aware model

In [ ]:
!python scripts/train.py \
    --config base.yaml data.yaml model_segformer.yaml train.yaml train_segformer.yaml \
    -o train.num_workers=2 -o train.epochs=40

## 4. Compare — baseline vs SegFormer+clDice
Watch IoU/Dice be similar while **clDice, connectivity-ratio and APLS** favour the
topology model — that is the USP, measured.

In [ ]:
!python scripts/evaluate.py \
    --checkpoint artifacts/checkpoints/baseline_unet_best.pth \
    --compare    artifacts/checkpoints/segformer_cldice_best.pth \
    --apls

## 5. Download the checkpoints
Place them in local `artifacts/checkpoints/` to run inference + the dashboard.

In [ ]:
try:
    from google.colab import files
    files.download('artifacts/checkpoints/segformer_cldice_best.pth')
    files.download('artifacts/checkpoints/baseline_unet_best.pth')
except Exception as e:
    print('Not on Colab or download unavailable:', e)